# Decision Trees

> Inspired by [Sam Lau](https://www.samlau.me/), who co-authored the [Learning Data Science](https://learningds.org/ch/12/pa_intro.html) book.

This lesson follows [Model Evaluation](model-evaluation.ipynb), where we used linear regression, train/test splits, and cross-validation to predict AQS measurements from PurpleAir sensor (PAS) measurements. By the end of this lesson, students will be able to:

- Describe the difference between classification and regression tasks.
- Apply `sklearn` to train a decision tree for classification and regression tasks.
- Compare the predictions of a decision tree and a linear regression model.

In [ ]:
import pandas as pd

## Classification versus regression

Everything we've seen so far fall under the category of **regression**, where we aim to predict a *numeric* target value (one column) from one or more features (one or more other columns). The other main category of problems is **classification**, which is just like regression except we aim to predict a *categorical* target value. For example, we might want to answer the question: **How can we predict whether a house belongs in `NY` or `SF` from its beds, baths, price, year of construction, square footage, price per square foot, and elevation?**

In [ ]:
homes = pd.read_csv("homes.csv")
homes

Let's learn about **decision trees**, a machine learning algorithm that can be used for classification (and also, as it turns out, regression too). Decision trees learn a nested if-then-else logical hierarchy to fit a training dataset. In the following visualization, the color and opacity of each box represents whether that subset of homes is more likely to be in `NY` or `SF`. The notes inside each node indicate information about the values:

1. The first line shows the condition. If the condition is true, go left; if not, go right.
1. The second line shows the percentage of samples represented by that node.
1. The third line shows the proportion of homes within that sample that belong in `["NY", "SF"]`.
1. The fourth line shows the majority class in that category, corresponding to the bigger number on line 3.

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree
X = homes.drop("city", axis=1)
y = homes["city"]
clf = DecisionTreeClassifier(max_depth=2).fit(X, y)

import matplotlib.pyplot as plt
plt.figure(dpi=300)
plot_tree(
    clf,
    feature_names=list(X.columns),
    class_names=["NY", "SF"],
    label="root",
    filled=True,
    impurity=False,
    proportion=True,
    rounded=False
)
display("")

In [ ]:
# And using default for arbitrary depth

from sklearn.tree import DecisionTreeClassifier, plot_tree
X = homes[["beds", "bath", "price", "year_built", "sqft", "price_per_sqft", "elevation"]]
y = homes["city"]
# Code for creating a new DecisionTreeClassifier specifies what we call "hyperparameters"
clf = DecisionTreeClassifier().fit(X, y)

import matplotlib.pyplot as plt
plt.figure(dpi=300)
plot_tree(
    clf,
    feature_names=X.columns,
    class_names=["NY", "SF"],
    label="root",
    filled=True,
    impurity=False,
    proportion=True,
    rounded=False
)
display("")

We can also use this dataset for regression too. Write a one-line expression to train a `DecisionTreeRegressor` model to predict the price of a home in this dataset from all other variables.

In [ ]:
from sklearn.tree import DecisionTreeRegressor

reg = ... # Use pd.get_dummies to convert categorical variables to one-hot encoding

reg.predict(pd.get_dummies(homes.drop("price", axis=1)))


Consider each of the following tasks and answer whether they would be best modeled as classification or regression.

<details>
<summary>Predict whether an email is spam or not spam.</summary>

Classification, since the target value is the category "spam" or "not spam".
</details>

<details>
<summary>Predict the number of views a video will receive based on subscriber count.</summary>

Regression, since the target value is a number.
</details>

<details>
<summary>Predict the next word to appear in a sentence.</summary>

Classification, since the target value is to choose from the dictionary of all possible next words.
</details>

> If you're curious about how ChatGPT works, Jay Mody has a good introduction to [GPT in 60 Lines of NumPy](https://jaykmody.com/blog/gpt-from-scratch/).

## Decision trees for regression: sensor calibration

In [Learning Algorithms](learning-algorithms.ipynb), we plotted the predictions for a linear regression model that was trained to take PAS measurements and predict AQS measurements. What do you think a decision tree model would look like for this simplified PurpleAir sensor calibration problem?

Here's a complete workflow for decision tree model evaluation using the practices from [Model Evaluation](model-evaluation.ipynb): a train/test split, recursive feature elimination, and grid search cross-validation. The resulting plot compares a linear regression model `lmplot` against the decisions made by a decision tree.

In [ ]:
import numpy as np
import seaborn as sns
from sklearn.feature_selection import RFE
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.tree import DecisionTreeRegressor

sensor_data = pd.read_csv("sensor_data.csv")

In [ ]:
# Split dataset into 80% training dataset and 20% testing dataset
X = sensor_data.drop("AQS", axis=1)
y = sensor_data["AQS"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Recursive feature elimination to select the single most important feature based on slope value
rfe = RFE(LinearRegression(), n_features_to_select=1, verbose=1)
rfe.fit(X_train, y_train)
# Print the best feature to predict AQS
rfe_feature = X.columns[rfe.ranking_.argmin()]
print("Best feature to predict AQS:", rfe_feature)
# Use only the best feature
X = X[[rfe_feature]]
X_train = X_train[[rfe_feature]]
X_test = X_test[[rfe_feature]]

# Grid search cross-validation to tune the max_depth hyperparameter using RMSE loss metric
search = GridSearchCV(
    estimator=DecisionTreeRegressor(),
    param_grid={"max_depth": [2, 3, 4, 5, 6, 7, 8, 9, 10]},
    scoring="neg_root_mean_squared_error",
    verbose=3,
)
search.fit(X_train, y_train)
# Print the best score and best estimator at the end of hyperparameter search
print("Mean score for best model:", search.best_score_)
reg = search.best_estimator_
print("Best model:", reg)

# During model evaluation, use the testing dataset
print("Test error:", root_mean_squared_error(y_test, search.predict(X_test)))

# Visualize the tree decisions compared to a LinearRegression model (lmplot)
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.tree import plot_tree
sns.set_theme()
grid = sns.lmplot(sensor_data, x=rfe_feature, y="AQS")
# Create a demonstration dataset that counts from 0 to the max PAS value
X_demo = pd.DataFrame(np.arange(X[rfe_feature].max()), columns=[rfe_feature])
ax = grid.facet_axis(0, 0)
ax.plot(X_demo, reg.predict(X_demo), c="orange", linewidth=3)
grid.set(title=f"lmplot vs {reg}")
# Legend: lmplot draws the blue regression line; we drew the orange decision tree line
ax.legend(handles=[
    plt.Line2D([], [], color="C0", linewidth=3, label="LinearRegression (lmplot)"),
    plt.Line2D([], [], color="orange", linewidth=3, label=f"{reg}"),
])
# Show nodes in the decision tree
plt.figure(dpi=300)
plot_tree(
    reg,
    max_depth=2, # Only show the first two levels
    feature_names=[rfe_feature],
    label="root",
    filled=True,
    impurity=False,
    proportion=True,
    rounded=False
);

The testing dataset error rates for both the `DecisionTreeRegressor` and the `LinearRegression` models are not too far apart. **Question:** Which model would you prefer to use? Justify your answer using either the error rates below or the visualizations above.

In [ ]:
print("Tree error:", root_mean_squared_error(y_test, search.predict(X_test)))
print("Line error:", root_mean_squared_error(
    y_test,
    LinearRegression().fit(X_train, y_train).predict(X_test),
))

*TODO*: Enter your response here

In Model Evaluation, we discussed how overfitting to the testing dataset can be mitigated by using cross-validation. But in answering the previous question on whether we should prefer a `DecisionTreeRegressor` model or a `LinearRegression` model, we've actually overfit the model by hand! **Question:** Explain how preferring one model over the other according to the visualization overfits to the testing dataset.

*TODO*: Enter your response here